In [1]:
from core import (
    skf,
    X_train,
    Y_train,
    evaluate_model,
    score
)

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import numpy as np
import joblib


scale_pos_weight_val = np.sum(Y_train == 0)/np.sum(Y_train == 1)

xgb = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric='auc',
    tree_method='hist',
    scale_pos_weight=scale_pos_weight_val
)

param_dist_xgb = {
    'n_estimators': [200, 500, 800, 1000, 1500],
    'learning_rate': [0.005, 0.008, 0.01, 0.02, 0.03],
    'max_depth': [4, 5, 6, 7],
    'min_child_weight': [5, 8, 10, 15],
    'gamma': [0.1, 0.2, 0.3],
    'reg_lambda': [5, 10, 15, 20],
    'reg_alpha': [0.5, 1, 2],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
}

random_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=200,
    cv=skf,
    scoring='roc_auc',
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_xgb.fit(X_train, Y_train)

best_xgb = random_xgb.best_estimator_

print(f'Лучшие параметры XGBoost: {random_xgb.best_params_}')
print(f'Лучшая ROC_AUC: {random_xgb.best_score_:.4f}')

results_xgb = evaluate_model(
    model=best_xgb,
    X=X_train,
    y=Y_train,
    cv=skf,
    scoring_dict=score
)

results_all = {}
results_all['XGBoost'] = results_xgb

df_results = pd.DataFrame.from_dict(results_all, orient='index').reset_index()
df_results = df_results.rename(columns={'index': 'Model'})

for col in ['Accuracy', 'F1', 'ROC-AUC']:
    if col in df_results.columns:
        df_results[col] = df_results[col].round(4)
print(df_results.to_markdown(index=False))


Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Лучшие параметры XGBoost: {'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 1500, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.02, 'gamma': 0.2, 'colsample_bytree': 0.7}
Лучшая ROC_AUC: 0.8871


ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.